# 🛡️ MayaGuard - LoRA Fine-Tuning: Claim Verification Classifier

This notebook fine-tunes a **DeBERTa-v3-base** model with **LoRA** adapters for
binary claim verification: given a `(claim, context)` pair, classify it as
`SUPPORTED` or `UNSUPPORTED`.

**Runtime:** Google Colab Free Tier (T4 GPU)
**Training Time:** ~15-30 minutes
**Base Model:** `microsoft/deberta-v3-base` (184M params)
**LoRA Rank:** 16, Alpha: 32

---

## 1. Install Dependencies

In [ ]:
!pip install -q peft transformers datasets accelerate evaluate scikit-learn torch

## 2. Upload Training Data

Run `python -m finetuning.data.generate_nli_dataset` locally first to generate
`train.jsonl` and `val.jsonl`, then upload them here.

In [ ]:
from google.colab import files
import os

os.makedirs("data", exist_ok=True)

print("Upload train.jsonl and val.jsonl files:")
uploaded = files.upload()

for name, content in uploaded.items():
    with open(f"data/{name}", "wb") as f:
        f.write(content)
    print(f"  Saved: data/{name} ({len(content)} bytes)")

## 3. Load and Inspect Dataset

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": "data/train.jsonl", "validation": "data/val.jsonl"},
)

# Map labels to integers
LABEL2ID = {"UNSUPPORTED": 0, "SUPPORTED": 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

def encode_label(example):
    example["label_id"] = LABEL2ID[example["label"]]
    return example

dataset = dataset.map(encode_label)

print(f"Train samples: {len(dataset['train'])}")
print(f"Val samples:   {len(dataset['validation'])}")
print(f"\nLabel distribution (train):")
print(f"  SUPPORTED:   {sum(1 for x in dataset['train'] if x['label_id'] == 1)}")
print(f"  UNSUPPORTED: {sum(1 for x in dataset['train'] if x['label_id'] == 0)}")
print(f"\nSample entry:")
print(dataset["train"][0])

## 4. Load Base Model and Tokenizer

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

MODEL_NAME = "microsoft/deberta-v3-base"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

# Count trainable parameters before LoRA
total_params = sum(p.numel() for p in model.parameters())
print(f"Base model parameters: {total_params:,}")

## 5. Apply LoRA Configuration

In [ ]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.SEQ_CLS,
    r=16,                         # LoRA rank
    lora_alpha=32,                # LoRA alpha (scaling factor)
    lora_dropout=0.1,
    target_modules=["query_proj", "value_proj"],  # DeBERTa attention layers
    bias="none",
)

model = get_peft_model(model, lora_config)

# Print trainable vs total parameters
model.print_trainable_parameters()

## 6. Tokenize Dataset

In [ ]:
def tokenize_fn(examples):
    tokenized = tokenizer(
        examples["claim"],
        examples["context"],
        truncation=True,
        max_length=512,
        padding="max_length",
    )
    tokenized["labels"] = examples["label_id"]
    return tokenized

tokenized_dataset = dataset.map(
    tokenize_fn,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

tokenized_dataset.set_format("torch")
print(f"Tokenized train shape: {tokenized_dataset['train'].shape}")
print(f"Tokenized val shape:   {tokenized_dataset['validation'].shape}")

## 7. Define Training Configuration

In [ ]:
import numpy as np
import evaluate
from transformers import TrainingArguments, Trainer

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_metric.compute(predictions=predictions, references=labels)
    f1 = f1_metric.compute(predictions=predictions, references=labels, average="binary")
    return {"accuracy": acc["accuracy"], "f1": f1["f1"]}


OUTPUT_DIR = "mayaguard-claim-verifier-lora"

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=10,
    fp16=True,
    report_to="none",
    save_total_limit=2,
)

print(f"Training configuration:")
print(f"  Epochs: {training_args.num_train_epochs}")
print(f"  Batch size: {training_args.per_device_train_batch_size}")
print(f"  Learning rate: {training_args.learning_rate}")
print(f"  FP16: {training_args.fp16}")

## 8. Train

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    compute_metrics=compute_metrics,
)

print("Starting LoRA fine-tuning...")
train_result = trainer.train()

print(f"\nTraining complete!")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")

## 9. Evaluate

In [ ]:
eval_results = trainer.evaluate()

print(f"\nValidation Results:")
print(f"  Accuracy: {eval_results['eval_accuracy']:.4f}")
print(f"  F1 Score: {eval_results['eval_f1']:.4f}")
print(f"  Loss:     {eval_results['eval_loss']:.4f}")

## 10. Save LoRA Adapter

In [ ]:
ADAPTER_DIR = "mayaguard-claim-verifier-lora/final_adapter"

# Save only the LoRA adapter weights (not the full model)
model.save_pretrained(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)

# Check adapter size
import os
adapter_size = sum(
    os.path.getsize(os.path.join(ADAPTER_DIR, f))
    for f in os.listdir(ADAPTER_DIR)
    if os.path.isfile(os.path.join(ADAPTER_DIR, f))
)
print(f"\nAdapter saved to: {ADAPTER_DIR}")
print(f"Adapter size: {adapter_size / 1024 / 1024:.2f} MB")
print(f"(vs base model ~700 MB - {adapter_size / (700 * 1024 * 1024) * 100:.1f}% of original)")

## 11. Test Inference

In [ ]:
import torch

def predict(claim: str, context: str) -> dict:
    """Run a single claim verification prediction."""
    inputs = tokenizer(claim, context, return_tensors="pt", truncation=True, max_length=512)
    inputs = {k: v.to(model.device) for k, v in inputs.items()}
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    probs = torch.softmax(outputs.logits, dim=-1)
    pred_class = torch.argmax(probs, dim=-1).item()
    confidence = probs[0][pred_class].item()
    
    return {
        "verdict": ID2LABEL[pred_class],
        "confidence": f"{confidence:.2%}",
        "claim": claim[:80],
    }


# Test with known examples
context = "Metformin is the primary pharmacotherapy for type 2 diabetes mellitus. It operates by activating AMPK in hepatocytes."

test_cases = [
    ("Metformin is used to treat type 2 diabetes.", "Should be SUPPORTED"),
    ("Metformin cures all forms of cancer.", "Should be UNSUPPORTED"),
    ("Metformin activates AMPK in hepatocytes.", "Should be SUPPORTED"),
    ("Metformin has no side effects.", "Should be UNSUPPORTED"),
]

print("\nInference Test Results:")
print("=" * 80)
for claim, expected in test_cases:
    result = predict(claim, context)
    status = "✅" if result["verdict"] in expected else "❌"
    print(f"{status} {result['verdict']} ({result['confidence']}) - {claim}")
    print(f"   Expected: {expected}")

## 12. Download Adapter

Download the adapter to your local machine, then place it in
`mayaguard/finetuning/outputs/claim_verifier_adapter/` and set
`FINETUNED_VERIFIER_ENABLED=true` in your `.env` file.

In [ ]:
import shutil

# Zip the adapter directory for download
shutil.make_archive("claim_verifier_adapter", "zip", ADAPTER_DIR)
files.download("claim_verifier_adapter.zip")

print("\nDownload complete! Next steps:")
print("1. Unzip to mayaguard/finetuning/outputs/claim_verifier_adapter/")
print("2. Set FINETUNED_VERIFIER_PATH=finetuning/outputs/claim_verifier_adapter")
print("3. Set FINETUNED_VERIFIER_ENABLED=true")
print("4. Restart the MayaGuard API server")

## 13. (Optional) Push to HuggingFace Hub

In [ ]:
# Uncomment and fill in your HuggingFace credentials to push the adapter
# from huggingface_hub import login
# login(token="hf_YOUR_TOKEN_HERE")
#
# model.push_to_hub("your-username/mayaguard-claim-verifier-lora")
# tokenizer.push_to_hub("your-username/mayaguard-claim-verifier-lora")
#
# print("Adapter pushed to HuggingFace Hub!")